In [1]:
from pyspark.sql import SparkSession
from minio import Minio
from pathlib import Path

In [2]:
DIMENSOES = [
    "usuarios",
    "planos",
    "artistas",
    "albuns",
    "musicas",
    "generos",
    "playlists",
    "dispositivos",
]

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

# Detecta se está rodando localmente (VS Code) ou dentro do Docker
is_docker = os.path.exists('/.dockerenv')

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost") if is_docker else "localhost"
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB   = os.getenv("POSTGRES_DB", "streaming")
POSTGRES_USER = os.getenv("POSTGRES_USER", "streaming")
POSTGRES_PASS = os.getenv("POSTGRES_PASSWORD", "streaming123")

MINIO_HOST = ("minio:9000" if is_docker else "localhost:9000")
MINIO_USER = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

spark = (
    SparkSession.builder
    .appName("landing-dimensoes")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

cliente_minio = Minio(
    MINIO_HOST,
    access_key=MINIO_USER,
    secret_key=MINIO_PASS,
    secure=False
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 16:48:32 WARN Utils: Your hostname, DESKTOP-S3JB25M, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/21 16:48:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/c/Users/felip/www/university/eng-dados/projeto-ed-final/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/felipe/.ivy2.5.2/cache
The jars for the packages stored in: /home/felipe/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-304c9e70-f0ca-4d88-a624-7a9627ebe332;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 381ms :: artifacts dl 4ms
	:: modules

In [4]:
BUCKETS_NECESSARIOS = ["landing", "bronze", "silver", "gold"]

buckets_existentes = {b.name for b in cliente_minio.list_buckets()}

for bucket in BUCKETS_NECESSARIOS:
    if bucket not in buckets_existentes:
        cliente_minio.make_bucket(bucket)
        print(f"Bucket criado: {bucket}")
    else:
        print(f"Bucket já existe: {bucket}")

Bucket criado: landing
Bucket criado: bronze
Bucket criado: silver
Bucket criado: gold


In [5]:
jdbc_url = f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

propriedades = {
    "user": POSTGRES_USER,
    "password": POSTGRES_PASS,
    "driver": "org.postgresql.Driver"
}

for tabela in DIMENSOES:
    print(f"\n{'=' * 50}")
    print(f"Processando tabela: {tabela}")
    print(f"{'=' * 50}")

    df = spark.read.jdbc(url=jdbc_url, table=tabela, properties=propriedades)

    total = df.count()
    print(f"Total de registros: {total}")

    caminho_local = f"/tmp/landing/{tabela}"

    df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(caminho_local)

    csv_dir = Path(caminho_local)
    arquivo_csv = next(
        arquivo for arquivo in csv_dir.iterdir()
        if arquivo.name.endswith(".csv")
    )

    cliente_minio.fput_object(
        "landing",
        f"{tabela}/{tabela}.csv",
        str(arquivo_csv)
    )

    print(f"{tabela}.csv enviado para o MinIO!")

print("\nIngestão das dimensões concluída com sucesso!")


Processando tabela: usuarios


Total de registros: 3000


usuarios.csv enviado para o MinIO!

Processando tabela: planos
Total de registros: 3
planos.csv enviado para o MinIO!

Processando tabela: artistas
Total de registros: 500
artistas.csv enviado para o MinIO!

Processando tabela: albuns
Total de registros: 2000
albuns.csv enviado para o MinIO!

Processando tabela: musicas
Total de registros: 10000
musicas.csv enviado para o MinIO!

Processando tabela: generos
Total de registros: 30
generos.csv enviado para o MinIO!

Processando tabela: playlists
Total de registros: 2000
playlists.csv enviado para o MinIO!

Processando tabela: dispositivos
Total de registros: 5000
dispositivos.csv enviado para o MinIO!

Ingestão das dimensões concluída com sucesso!


In [6]:
spark.stop()